# Partition Exploration Notebook

Interactive analysis of $p = 2^m + q^n$ decompositions over ~1.17 billion primes.

**Data access**: DuckDB backend (partition_counts table + decompositions parquet view).
All heavy computation stays in DuckDB or Polars streaming -- nothing materializes the full dataset.

**Charts**: Altair + VegaFusion for server-side aggregation. Charts can handle millions of data points.

In [1]:
import sys, os

# Ensure working directory is project root (not notebooks/)
# so pixi.toml and relative data paths resolve correctly.
_project_root = os.path.dirname(os.path.abspath(os.getcwd()))
os.chdir(_project_root)
sys.path.insert(0, os.path.join(_project_root, "src"))

import altair as alt
import polars as pl
import duckdb

# VegaFusion for server-side aggregation of large datasets.
# Default html renderer works; data is compiled into the output.
alt.data_transformers.enable("vegafusion")

# Connect to DuckDB (read-only, persistent for this session)
from funbuns.querydb import QueryDB, get_db_path

db = QueryDB(read_only=True)
db.__enter__()
conn = db.conn

print(f"Database: {db.db_path}")
print(f"Size: {db.db_path.stat().st_size / (1024**3):.2f} GB")

# Quick stats
prime_count = conn.execute("SELECT COUNT(*) FROM partition_counts").fetchone()[0]
max_p = conn.execute("SELECT MAX(p) FROM partition_counts").fetchone()[0]
print(f"Primes indexed: {prime_count:,}")
print(f"Largest prime: {max_p:,}")

Database: /home/erpage159/fluid/research/funbuns/data/funbuns.duckdb
Size: 10.48 GB
Primes indexed: 1,171,040,000
Largest prime: 26,900,456,263


## Helper: SQL to DataFrame

`sql()` runs a query via DuckDB and returns a Polars DataFrame. For large result sets, use `sql_lazy()` which returns a DuckDB relation you can further filter before collecting.

In [3]:
def sql(query: str) -> pl.DataFrame:
    """Run SQL via DuckDB, return Polars DataFrame."""
    return conn.execute(query).pl()

def sql_arrow(query: str):
    """Run SQL via DuckDB, return PyArrow Table (for Altair/VegaFusion)."""
    return conn.execute(query).arrow()

def decomp_sample(where: str = "1=1", limit: int = 100_000) -> pl.DataFrame:
    """Sample decomposition rows with optional filter.
    
    Examples:
        decomp_sample("q_k = 3")
        decomp_sample("m_k % 2 = 0 AND n_k < 10")
        decomp_sample("q_k IN (3, 5, 7) AND p < 10^9", limit=500_000)
    """
    return sql(f"""
        SELECT p, m_k, q_k, n_k
        FROM decompositions
        WHERE q_k > 0 AND {where}
        LIMIT {limit}
    """)

## Example 1: k-distribution

In [4]:
k_dist = sql("""
    SELECT k, COUNT(*) AS count
    FROM partition_counts
    GROUP BY k ORDER BY k
""")

alt.Chart(k_dist).transform_filter(
    alt.datum.count > 0
).mark_bar().encode(
    x=alt.X("k:O", title="decomposition count k"),
    y=alt.Y("count:Q", title="number of primes",
            stack=None,
            scale=alt.Scale(type="log")),
    tooltip=["k", "count"]
).properties(width=500, title="Distribution of decomposition counts")

alt.Chart(...)

## Example 2: Modular arithmetic over decompositions

Aggregate decomposition rows in DuckDB, chart the result.
Use `partition_counts` to scope the query first -- scanning the full decompositions view is expensive.

In [7]:
# n mod 4 distribution for q=3 among k=5 primes
# Scoped via partition_counts first, then join decompositions
n_mod4 = sql("""
    SELECT d.n_k % 4 AS n_mod_4, COUNT(*) AS count
    FROM decompositions d
    JOIN partition_counts pc ON d.p = pc.p
    WHERE pc.k = 3 AND d.q_k = 3
    GROUP BY 1
    ORDER BY 1
""")
display(n_mod4)

alt.Chart(n_mod4).mark_bar().encode(
    x=alt.X("n_mod_4:O", title="n mod 4"),
    y=alt.Y("count:Q", title="decompositions", stack=None),
    tooltip=["n_mod_4", "count"]
).properties(width=300, title="n mod 4 for q=3, k=3 primes")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

n_mod_4,count
i64,i64
0,13
1,12
2,14
3,8


alt.Chart(...)

## Example 3: Obstruction grid heatmap

For a given small prime $\ell$, the residues $(m \bmod \mathrm{ord}_\ell(2),\; n \bmod \mathrm{ord}_\ell(q))$ determine local solvability.
This computes the empirical hit counts on that grid for q=3, using DuckDB for the full aggregation.

In [ ]:
from sage.all import Mod

def obstruction_grid(q: int, ell: int, p_max: int = None):
    """Compute the (m mod ord_ell(2), n mod ord_ell(q)) hit-count grid.
    
    Returns a DataFrame with columns: m_res, n_res, count
    """
    p_clause = f"AND p <= {p_max}" if p_max else ""
    
    # Multiplicative orders via SageMath (PARI-backed)
    ord_2 = int(Mod(2, ell).multiplicative_order())
    ord_q = int(Mod(q, ell).multiplicative_order())
    
    print(f"ell={ell}: ord_{ell}(2) = {ord_2}, ord_{ell}({q}) = {ord_q}, grid = {ord_2} x {ord_q}")
    
    return sql(f"""
        SELECT m_k % {ord_2} AS m_res,
               n_k % {ord_q} AS n_res,
               COUNT(*) AS count
        FROM decompositions
        WHERE q_k = {q} AND q_k > 0 {p_clause}
        GROUP BY 1, 2
        ORDER BY 1, 2
    """)

# Example: q=3, ell=5 (ord_5(2)=4, ord_5(3)=4 -> 4x4 grid)
grid = obstruction_grid(q=3, ell=5)
display(grid)

alt.Chart(grid).mark_rect().encode(
    x=alt.X("m_res:O", title="m mod ord(2)"),
    y=alt.Y("n_res:O", title="n mod ord(3)"),
    color=alt.Color("count:Q", scale=alt.Scale(scheme="viridis")),
    tooltip=["m_res", "n_res", "count"]
).properties(width=250, height=250, title="Obstruction grid: q=3, ell=5")

## Example 4: Large-scale scatter with VegaFusion

VegaFusion handles the server-side aggregation -- you can pass hundreds of thousands of rows to Altair and it bins/aggregates before sending to the browser.

In [ ]:
# m vs n scatter for q=3, first 200k decompositions
# VegaFusion bins this server-side before rendering
scatter_data = decomp_sample("q_k = 3", limit=200_000)

alt.Chart(scatter_data).mark_circle(size=3, opacity=0.3).encode(
    x=alt.X("m_k:Q", title="m"),
    y=alt.Y("n_k:Q", title="n"),
    tooltip=["p", "m_k", "n_k"]
).properties(width=600, height=400, title="(m, n) pairs for q=3 (200k sample)")

## Example 5: Custom modular aggregation

Template for running arbitrary modular arithmetic across filtered subsets.
Change the modulus, filter, and grouping to explore different structures.

In [ ]:
def mod_distribution(column: str, modulus: int, where: str = "q_k > 0"):
    """Count decomposition rows grouped by (column mod modulus).
    
    Examples:
        mod_distribution("m_k", 12, "q_k = 3")
        mod_distribution("p", 6)
        mod_distribution("n_k", 3, "q_k = 7 AND m_k % 2 = 0")
    """
    return sql(f"""
        SELECT {column} % {modulus} AS residue, COUNT(*) AS count
        FROM decompositions
        WHERE {where}
        GROUP BY 1
        ORDER BY 1
    """)

# Example: m mod 12 for q=3 (relates to ord_13(2) = 12)
m12 = mod_distribution("m_k", 12, "q_k = 3")
display(m12)

alt.Chart(m12).mark_bar().encode(
    x=alt.X("residue:O", title="m mod 12"),
    y=alt.Y("count:Q"),
    tooltip=["residue", "count"]
).properties(width=400, title="m mod 12 for q=3 (full dataset)")

## Example 6: q-frequency by prime range

How does the frequency of different q bases change as primes get larger?
DuckDB bins p into ranges and counts q occurrences -- full dataset, one SQL query.

In [ ]:
# Top-5 q bases by prime range (binned into 500M intervals)
q_by_range = sql("""
    SELECT FLOOR(p / 500000000) * 500000000 AS p_bin,
           q_k,
           COUNT(*) AS count
    FROM decompositions
    WHERE q_k > 0 AND q_k <= 20
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""")

alt.Chart(q_by_range).mark_line().encode(
    x=alt.X("p_bin:Q", title="prime range (lower bound)", axis=alt.Axis(format="~s")),
    y=alt.Y("count:Q", title="decompositions"),
    color=alt.Color("q_k:N", title="q"),
    tooltip=["p_bin", "q_k", "count"]
).properties(width=700, height=350, title="q frequency by prime range (q <= 20)")

## Polynomial Templates and Chain Expansion

Each decomposition $p = 2^m + q^n$ defines a polynomial template $f_{m,n}(X) = 2^m + X^n$.
If $q$ itself decomposes as $q = 2^{m'} + q'^{n'}$, substituting gives a **chain**:

$$p = f_{m,n}(f_{m',n'}(q')) = 2^m + (2^{m'} + q'^{n'})^n$$

For $n=1$ this collapses: $p = 2^m + 2^{m'} + q'$.

We enumerate all templates up to the first totally obstructed prime (149),
build maximal chains by recursive substitution, and derive the resulting
symbolic expressions. These can then be evaluated over $\mathbb{F}_p$.

In [ ]:
import itertools
from collections import defaultdict

# Use decompositions_up_to() — k comes from partition_counts (DuckDB), not recomputed
all_data = db.decompositions_up_to(149)

# k per prime (single-level dict from 2-column unique)
k_by_p = dict(all_data.select('p', 'k').unique(subset=['p']).iter_rows())

# Decomposition lookup for chain expansion: p -> [(m, q, n)]
decomp_map = defaultdict(list)
for row in all_data.filter(pl.col('q').is_not_null()).iter_rows(named=True):
    decomp_map[row['p']].append((row['m'], row['q'], row['n']))

# Obstructed primes: k=0 directly from DuckDB partition_counts
obstructed = sorted(p for p, k in k_by_p.items() if k == 0)

print(f"Primes loaded: {len(k_by_p)} ({len(decomp_map)} with decompositions)")
print(f"Obstructed (k=0): {obstructed}")

# Distinct (m, n) templates — flatten via itertools
templates = sorted(set(
    (m, n)
    for m, _, n in itertools.chain.from_iterable(decomp_map.values())
))
print(f"\n{len(templates)} distinct templates f_{{m,n}}(q) = 2^m + q^n:")
for m, n in templates:
    print(f"  f_{{{m},{n}}} = {2**m} + q^{n}")

### Symbolic polynomial ring

Work in $\mathbb{Z}[q]$ using SageMath. Each template $f_{m,n}(q) = 2^m + q^n$ is a polynomial.
Chain composition $f \circ g$ substitutes the inner polynomial for $q$.

In [ ]:
from sage.all import ZZ, PolynomialRing, GF

R = PolynomialRing(ZZ, 'q')
q = R.gen()

def template(m, n):
    """Polynomial template f_{m,n}(q) = 2^m + q^n."""
    return ZZ(2)**m + q**n

# Template polynomials + reverse lookup (poly -> (m,n))
template_polys = {(m, n): template(m, n) for m, n in templates}
poly_to_template = {f: (m, n) for (m, n), f in template_polys.items()}

for (m, n), f in template_polys.items():
    print(f"f_{{{m},{n}}}(q) = {f}")
print(f"\n{len(template_polys)} templates")

### Chain expansion (memoized DAG)

Since $q < p$ always ($p = 2^m + q^n$, so $q^n < p$), the decomposition graph is a DAG
with edges pointing to strictly smaller primes. Processing in ascending order is
topological: each prime's full subtree is expanded **exactly once** and cached.

Shared subtrees (e.g., multiple primes routing through $q = 3$) are computed once.

In [ ]:
def _compose_chains(m, n, q_val, sub_chains):
    """Compose template f_{m,n} with pre-computed sub-chains from q_val."""
    f = template(m, n)
    return [
        (tq, [(m, n, q_val)] + sub_st, f(sub_poly))
        for tq, sub_st, sub_poly in sub_chains
    ]


def build_chain_memo(decomp_map):
    """Bottom-up memoized chain expansion over the decomposition DAG.
    
    q < p always (p = 2^m + q^n), so ascending prime order is topological.
    Each prime's subtree is expanded exactly once and reused.
    
    Returns: dict mapping p -> [(terminal_q, steps, polynomial)]
    """
    memo = {}

    for p in sorted(decomp_map):
        chains = []
        for m, q_val, n in decomp_map[p]:
            # q_val < p, so memo[q_val] is already computed if it has decompositions.
            # If not in memo, q_val is terminal (obstructed or out of range).
            sub = memo.get(q_val, [(q_val, [], q)])
            chains.extend(_compose_chains(m, n, q_val, sub))
        memo[p] = chains

    return memo


chain_memo = build_chain_memo(decomp_map)

# Example: chains from p=7
print("Chains from p = 7:")
print("=" * 60)
for terminal, steps, poly in chain_memo.get(7, []):
    step_str = " -> ".join(f"2^{m}+q^{n} [q={qv}]" for m, n, qv in steps)
    print(f"  terminal q={terminal}")
    print(f"  chain: {step_str}")
    print(f"  poly:  {poly}")
    print(f"  check: {poly(terminal)} == 7? {poly(terminal) == 7}")
    print()

### Full chain catalog

Enumerate all maximal chains for every prime up to 149 that has at least one decomposition.
Collect the unique composed polynomials and identify which primes share the same polynomial structure.

In [ ]:
# Flatten memo into DataFrame + poly_lookup (single pass)
poly_lookup = {}   # str(poly) -> SageMath polynomial
flat_rows = []

for p, chains in sorted(chain_memo.items()):
    k = k_by_p[p]
    for tq, st, poly in chains:
        ps = str(poly)
        flat_rows.append((p, k, tq, len(st), ps, poly.degree()))
        poly_lookup[ps] = poly

chain_df = pl.DataFrame(
    flat_rows,
    schema=["source_p", "k", "terminal_q", "depth", "poly", "degree"],
    orient="row",
)

print(f"Total chains: {len(chain_df)}")
print(f"Unique polynomials: {len(poly_lookup)}")
display(chain_df)

In [ ]:
# Polynomials shared by multiple primes — grouped by k (Polars)
shared = (
    chain_df
    .group_by("poly")
    .agg(
        pl.col("source_p").unique().sort().alias("primes"),
        pl.col("source_p").n_unique().alias("n_primes"),
        pl.col("k").first().alias("k"),
    )
    .filter(pl.col("n_primes") > 1)
    .sort("n_primes", descending=True)
)
print("Polynomials shared by multiple primes:")
display(shared)

# Recurrence relations: f o g collapses to a single template?
# O(1) lookup via poly_to_template instead of linear scan
print("\nComposition identities (f o g = h as single template):")
print("=" * 60)
identities = []
for ((m1, n1), f1), ((m2, n2), f2) in itertools.product(template_polys.items(), repeat=2):
    composed = f1(f2)
    if composed in poly_to_template:
        m3, n3 = poly_to_template[composed]
        identities.append(((m1, n1), (m2, n2), (m3, n3), composed))
        print(f"  f_{{{m1},{n1}}} o f_{{{m2},{n2}}} = f_{{{m3},{n3}}}  ({composed})")

if not identities:
    print("  (none — compositions produce higher-degree polynomials)")

# Polynomial GCD between composed polynomials — shared factors = shared root structure
print("\nNon-trivial GCDs between composed polynomials:")
print("=" * 60)
poly_items = list(poly_lookup.items())
gcd_hits = []
for (ps1, p1), (ps2, p2) in itertools.combinations(poly_items, 2):
    g = p1.gcd(p2)
    if g.degree() > 0:
        gcd_hits.append((ps1, ps2, str(g), g.degree()))

for ps1, ps2, g_str, deg in sorted(gcd_hits, key=lambda x: -x[3]):
    print(f"  gcd({ps1}, {ps2}) = {g_str}")

if not gcd_hits:
    print("  (all pairwise GCDs are constant — polynomials are coprime)")

# Polynomial division: which composed polynomials are divisible by a template?
print("\nTemplate divisibility (composed = template * quotient):")
print("=" * 60)
div_hits = []
for (ps, poly), ((m, n), tmpl) in itertools.product(poly_lookup.items(), template_polys.items()):
    if poly == tmpl or poly.degree() <= tmpl.degree():
        continue
    quo, rem = poly.quo_rem(tmpl)
    if rem == 0:
        div_hits.append((ps, m, n, str(quo)))

for ps, m, n, quo_str in div_hits:
    print(f"  ({ps}) = f_{{{m},{n}}} * ({quo_str})")

if not div_hits:
    print("  (no composed polynomial is divisible by a single template)")

### Evaluation over prime fields

Given a composed polynomial $f \in \mathbb{Z}[q]$, reduce modulo $p$ and find roots in $\mathbb{F}_p$.
A chain is **locally valid** at $p$ if $f(q) \equiv 0 \pmod{p}$ for some $q$.

For obstructed primes (k=0), no single template has a root. But composed polynomials
from chains through smaller primes might still have roots -- revealing structure
that single-step decomposition misses.

In [ ]:
def roots_in_GF(poly, p):
    """Find roots of poly in GF(p). Returns list of integer roots."""
    Fp = GF(p)
    Rp = PolynomialRing(Fp, 'x')
    return [ZZ(r) for r, _ in Rp(poly).roots()]


def eval_polys_at(target_p, poly_lookup):
    """Evaluate all polynomials in poly_lookup over GF(target_p).
    
    Single loop over the flat poly_lookup dict. Returns list of
    (poly_str, roots) for polynomials that have roots.
    """
    Fp = GF(target_p)
    Rp = PolynomialRing(Fp, 'x')
    hits = []
    for ps, poly in poly_lookup.items():
        roots = [ZZ(r) for r, _ in Rp(poly).roots()]
        if roots:
            hits.append((ps, roots))
    return hits

In [ ]:
# For each obstructed prime: how many composed polynomials have roots?
# Single loop over obstructed primes, eval_polys_at handles the flat iteration.
print("Obstructed primes: GF(p) root structure")
print("=" * 60)

obs_results = []
for obs_p in obstructed:
    hits = eval_polys_at(obs_p, poly_lookup)
    obs_results.append((obs_p, len(hits), hits[:5]))
    print(f"\np = {obs_p}: {len(hits)}/{len(poly_lookup)} polynomials have roots in GF({obs_p})")
    for ps, roots in hits[:5]:
        print(f"  {ps}  ->  roots: {[int(r) for r in roots]}")
    if len(hits) > 5:
        print(f"  ... and {len(hits) - 5} more")

### Chain depth and degree structure

Summary statistics on how chains grow: maximum depth, polynomial degree distribution, and which templates dominate at each depth level.

In [ ]:
# Chain statistics — pure Polars

print("Chains by source prime k (decomposition count from DuckDB):")
display(chain_df.group_by("k").agg(
    pl.len().alias("chains"),
    pl.col("source_p").n_unique().alias("primes"),
    pl.col("depth").max().alias("max_depth"),
    pl.col("degree").max().alias("max_degree"),
).sort("k"))

print("\nChain depth distribution:")
display(chain_df.group_by("depth").len().sort("depth"))

print("\nTerminal q distribution:")
display(
    chain_df
    .group_by("terminal_q")
    .len()
    .sort("terminal_q")
    .with_columns(
        pl.when(pl.col("terminal_q").is_in(obstructed))
        .then(pl.lit("OBSTRUCTED"))
        .when(pl.col("terminal_q") > 149)
        .then(pl.lit("OUT OF RANGE"))
        .otherwise(pl.lit(""))
        .alias("status")
    )
)

## Scratch space

Use the cells below for ad-hoc queries. The `sql()`, `decomp_sample()`, `mod_distribution()`, and `obstruction_grid()` helpers are available.

**SageMath**: Switch to the `sagemath` kernel (Kernel > Change Kernel) for number-theoretic computations. The DuckDB connection works in either kernel.